# 🖼️ Лабораторная работа 5. MNIST

Цель: обучить первую модель на настоящем Dataset изображений.

Мы разберём `Dataset`, `DataLoader`, Batch, Flatten, `CrossEntropyLoss`, Accuracy, Train/Test и сохранение модели.


# 1. Импорт библиотек

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

RANDOM_SEED = 42
BATCH_SIZE = 64
EPOCHS = 3
LEARNING_RATE = 0.001

torch.manual_seed(RANDOM_SEED)

print("PyTorch:", torch.__version__)

# 2. Путь для Dataset

In [ ]:
DATA_DIR = Path("../../Datasets/mnist")
print(DATA_DIR)

# 3. Загружаем MNIST

In [ ]:
transform = transforms.ToTensor()

train_dataset = datasets.MNIST(
    root=DATA_DIR,
    train=True,
    download=True,
    transform=transform,
)

test_dataset = datasets.MNIST(
    root=DATA_DIR,
    train=False,
    download=True,
    transform=transform,
)

print("Train:", len(train_dataset))
print("Test:", len(test_dataset))

# 4. Смотрим один пример

In [ ]:
image, target = train_dataset[0]

print("Target:", target)
print("Image shape:", image.shape)
print("dtype:", image.dtype)
print("min:", image.min().item())
print("max:", image.max().item())

# 5. Показываем изображение

In [ ]:
plt.figure(figsize=(4, 4))
plt.imshow(image.squeeze(), cmap="gray")
plt.title(f"Target: {target}")
plt.axis("off")
plt.show()

# 6. Несколько цифр

In [ ]:
plt.figure(figsize=(10, 4))

for i in range(10):
    image, target = train_dataset[i]
    plt.subplot(2, 5, i + 1)
    plt.imshow(image.squeeze(), cmap="gray")
    plt.title(str(target))
    plt.axis("off")

plt.tight_layout()
plt.show()

# 7. DataLoader

In [ ]:
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
)

print("Train batches:", len(train_loader))
print("Test batches:", len(test_loader))

# 8. Один Batch

In [ ]:
images, targets = next(iter(train_loader))

print("Images shape:", images.shape)
print("Targets shape:", targets.shape)
print("Первые targets:", targets[:10])

# 9. Flatten

In [ ]:
flatten = nn.Flatten()
flat_images = flatten(images)

print("До:", images.shape)
print("После:", flat_images.shape)

# 10. Модель

In [ ]:
class MNISTNetwork(nn.Module):
    def __init__(self):
        super().__init__()

        self.network = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28 * 28, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 10),
        )

    def forward(self, images):
        return self.network(images)


model = MNISTNetwork()
print(model)

# 11. Количество параметров

In [ ]:
total_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
)

print("Всего параметров:", total_parameters)

# 12. Forward Pass

In [ ]:
logits = model(images)

print("Input shape:", images.shape)
print("Logits shape:", logits.shape)
print("Первый набор logits:")
print(logits[0])

# 13. argmax

In [ ]:
predictions = logits.argmax(dim=1)

print("Predictions:", predictions[:10])
print("Targets:", targets[:10])

# 14. CrossEntropyLoss

In [ ]:
loss_function = nn.CrossEntropyLoss()
loss = loss_function(logits, targets)

print("Loss:", loss.item())

# 15. Optimizer

In [ ]:
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE,
)

print(optimizer)

# 16. Одна Epoch обучения

In [ ]:
def train_one_epoch(model, loader, loss_function, optimizer, device):
    model.train()

    total_loss = 0.0
    correct = 0
    total = 0

    for images, targets in loader:
        images = images.to(device)
        targets = targets.to(device)

        logits = model(images)
        loss = loss_function(logits, targets)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        predictions = logits.argmax(dim=1)
        correct += (predictions == targets).sum().item()
        total += targets.size(0)

    return total_loss / len(loader), correct / total

# 17. Оценка

In [ ]:
def evaluate(model, loader, loss_function, device):
    model.eval()

    total_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, targets in loader:
            images = images.to(device)
            targets = targets.to(device)

            logits = model(images)
            loss = loss_function(logits, targets)

            total_loss += loss.item()

            predictions = logits.argmax(dim=1)
            correct += (predictions == targets).sum().item()
            total += targets.size(0)

    return total_loss / len(loader), correct / total

# 18. Device

In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = model.to(device)

print("Device:", device)

# 19. Обучение

In [ ]:
history = {
    "train_loss": [],
    "train_accuracy": [],
    "test_loss": [],
    "test_accuracy": [],
}

for epoch in range(EPOCHS):
    train_loss, train_accuracy = train_one_epoch(
        model,
        train_loader,
        loss_function,
        optimizer,
        device,
    )

    test_loss, test_accuracy = evaluate(
        model,
        test_loader,
        loss_function,
        device,
    )

    history["train_loss"].append(train_loss)
    history["train_accuracy"].append(train_accuracy)
    history["test_loss"].append(test_loss)
    history["test_accuracy"].append(test_accuracy)

    print(
        f"Epoch {epoch + 1}/{EPOCHS} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_accuracy:.4f} | "
        f"Test Loss: {test_loss:.4f} | "
        f"Test Acc: {test_accuracy:.4f}"
    )

# 20. График Loss

In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(history["train_loss"], label="Train")
plt.plot(history["test_loss"], label="Test")
plt.title("MNIST Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)
plt.show()

# 21. График Accuracy

In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(history["train_accuracy"], label="Train")
plt.plot(history["test_accuracy"], label="Test")
plt.title("MNIST Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(True)
plt.show()

# 22. Предсказания

In [ ]:
model.eval()

images, targets = next(iter(test_loader))
images_device = images.to(device)

with torch.no_grad():
    logits = model(images_device)
    predictions = logits.argmax(dim=1).cpu()

plt.figure(figsize=(12, 6))

for i in range(12):
    plt.subplot(3, 4, i + 1)
    plt.imshow(images[i].squeeze(), cmap="gray")
    plt.title(f"T={targets[i].item()} / P={predictions[i].item()}")
    plt.axis("off")

plt.tight_layout()
plt.show()

# 23. Сохраняем модель

In [ ]:
MODEL_PATH = Path("mnist_model.pth")

torch.save(
    model.state_dict(),
    MODEL_PATH,
)

print("Сохранено:", MODEL_PATH.resolve())

# 24. Загружаем модель

In [ ]:
loaded_model = MNISTNetwork().to(device)

loaded_model.load_state_dict(
    torch.load(
        MODEL_PATH,
        map_location=device,
    )
)

loaded_model.eval()

loaded_loss, loaded_accuracy = evaluate(
    loaded_model,
    test_loader,
    loss_function,
    device,
)

print("Loaded model Test Accuracy:", loaded_accuracy)

# 25. 📌 Что нужно запомнить

```text
Dataset
↓
DataLoader
↓
Batch
↓
Flatten
↓
Model
↓
10 logits
↓
CrossEntropyLoss
↓
Backward
↓
Optimizer
```


# 26. 🧩 Эксперименты

Попробуй:

- увеличить `EPOCHS`;
- изменить `BATCH_SIZE`;
- изменить скрытые размеры;
- заменить Adam на SGD;
- сравнить большую и маленькую модель.


# 27. ➡️ Следующая глава

# Глава 6. CNN — свёрточные нейронные сети
